# Taller N.º 7 — Spark SQL: consultas y planes de ejecución

**Sesión 24 · Big Data 2026-II · Universidad Externado de Colombia**

**Carácter del taller:** formativo. El formato —ejecución con inteligencia artificial permitida, corte, cuestionario sin inteligencia artificial y evidencia en capturas— es exactamente el que usará el Examen II.

**Qué cambia respecto del Taller 6**

1. Se entregan **cuatro capturas de pantalla obligatorias** (C1 a C4) en un archivo en PDF, que se entrega en el corte.
2. La hoja de registro incorpora una columna **N.º de captura**, para vincular cada valor con la evidencia que lo respalda.
3. Las reglas de material de la Fase 2 son las **definitivas del Examen II**. Se aplicarán sin cambios en el Taller 8 y en el examen.

---
## Antes de empezar (1): cómo se toma una captura válida

Se exigen únicamente cuatro capturas. Una captura adicional no otorga puntos, y la ausencia de una obligatoria deja sin respaldo las preguntas que dependan de ella.

| N.º | Ejecución | Qué debe verse |
|---|---|---|
| **C1** | E2 | Salida de `explain(True)`: basta la sección `== Physical Plan ==`, con la línea `FileScan` y los nodos `Exchange` visibles |
| **C2** | E3 | Pestaña **SQL / DataFrame** del portal: detalle de la consulta ejecutada, con el plan final y los nodos `AQEShuffleRead` |
| **C3** | E4 | Pestaña **SQL / DataFrame** del portal: detalle de la consulta ajustada |
| **C4** | E5 | Salida de `explain()` de las dos consultas, con las dos líneas `FileScan` visibles |

**Criterio de suficiencia**

- En las capturas del **portal** (C2 y C3) debe verse la **barra de direcciones** del navegador. La URL contiene el identificador `application_…`, que es único por clúster y acredita que la ejecución es propia.
- En las capturas de **salidas del notebook** (C1 y C4) debe verse la celda completa, incluida la línea `sc.setJobGroup(...)` o la consulta que la produce.
- Cada captura se numera C1, C2, C3 o C4 en el archivo de Word, en ese orden.
- Cada captura debe presentar la fecha y hora del sistema.

---
## Antes de empezar (2): cómo ubicar cada ejecución en el portal

Igual que en el Taller 6, cada celda de código incluye una línea `sc.setJobGroup("E#", "...")`. Esa etiqueta aparece en la columna **Description**, tanto en las pestañas *Jobs* y *Stages* como en la pestaña **SQL / DataFrame**.

La diferencia entre las dos pestañas está en qué representa cada fila:

- En *Jobs*, cada fila es un **job**. Una misma ejecución del notebook puede ocupar varias filas.
- En *SQL / DataFrame*, cada fila es una **consulta completa**, y su columna **Job IDs** enumera todos los jobs que la compusieron.

Por eso las capturas C2 y C3 se toman en *SQL / DataFrame*: allí cada consulta aparece una sola vez, con su etiqueta y con la lista de sus Job ID.

Conviene recordar, del Taller 6, que **una sola acción genera varios Job ID** cuando hay más de un *shuffle*. Ese es el comportamiento de AQE.

---
## Ejecución 1 — Sesión, datos y vistas temporales

Se crea la conexión con el clúster, se lee el archivo desde HDFS y se registra como vista temporal, que es lo que permite consultarlo con sintaxis SQL (Sesión 23, bloque 2).

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Taller7 - Spark SQL y planes de ejecucion")
    .master("yarn")
    .getOrCreate()
)

sc = spark.sparkContext

print("Application ID:", sc.applicationId)
print("Version de Spark:", spark.version)
print("AQE habilitado:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Particiones de shuffle por defecto:", spark.conf.get("spark.sql.shuffle.partitions"))

Application ID: application_1790370498259_0001
Version de Spark: 3.5.7
AQE habilitado: true
Particiones de shuffle por defecto: 200


In [2]:
sc.setJobGroup("E1", "E1 - Lectura del CSV y registro de la vista temporal")

df = spark.read.csv(
    "hdfs:///datasets/global_power_plant_database.csv",
    header=True,
    inferSchema=True
)

df.createOrReplaceTempView("plantas")

print("Filas:", df.count(), "| Columnas:", len(df.columns))
print("Vistas registradas:", [t.name for t in spark.catalog.listTables()])

Filas: 34936 | Columnas: 36
Vistas registradas: ['plantas']


La Ejecución 5 compara el mismo cálculo sobre CSV y sobre Parquet. El directorio Parquet se creó en la Sesión 23 y persiste en el HDFS propio. La siguiente celda verifica que exista y, si no está, lo genera.

In [3]:
sc.setJobGroup("E1b", "E1b - Verificacion de la vista Parquet")

ruta_parquet = "hdfs:///datasets/plantas_parquet"

try:
    # Si el directorio no existe, la linea siguiente lanza una excepcion.
    df_parquet = spark.read.parquet(ruta_parquet)
    print("El directorio Parquet ya existe. Se reutiliza.")
except Exception:
    print("No se encontro el directorio Parquet. Se genera ahora (toma unos segundos).")
    df.write.parquet(ruta_parquet, mode="overwrite")
    df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.createOrReplaceTempView("plantas_parquet")
print("Vistas registradas:", [t.name for t in spark.catalog.listTables()])

El directorio Parquet ya existe. Se reutiliza.
Vistas registradas: ['plantas', 'plantas_parquet']


**Registrar en la hoja (fila 1):** Application ID · número de executors vivos, sin contar el `driver` · cores por executor · valor de `spark.sql.shuffle.partitions`.

*Fuente:* salida de la primera celda y pestaña **Executors** del portal.

*No requiere captura.*

---
## Ejecución 2 — El plan antes de ejecutar

La siguiente consulta responde a la pregunta analítica: cuál es la capacidad hidroeléctrica instalada por país. Por ahora **no se ejecuta**: solo se pide su plan con `explain(True)`, que muestra las cuatro fases del Catalyst Optimizer vistas en la Sesión 23.

Al leer el plan físico conviene fijarse en tres cosas:

- **`ReadSchema`** dentro de `FileScan`: cuántas columnas se van a leer realmente, de las 36 que tiene el archivo (*projection pruning*).
- **`PushedFilters`**: qué condiciones del `WHERE` se entregan al lector del archivo (*predicate pushdown*).
- **`Exchange`**: cada uno de estos nodos es un *shuffle*, es decir, movimiento de datos entre executors. Interesa cuántos hay y de qué tipo (`hashpartitioning` o `rangepartitioning`).

In [4]:
consulta = spark.sql("""
    SELECT country_long,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    WHERE primary_fuel = 'Hydro'
    GROUP BY country_long
    ORDER BY capacidad_total_mw DESC
""")

consulta.explain(True)

== Parsed Logical Plan ==
'Sort ['capacidad_total_mw DESC NULLS LAST], true
+- 'Aggregate ['country_long], ['country_long, 'COUNT(1) AS num_plantas#266, 'ROUND('SUM('capacity_mw), 2) AS capacidad_total_mw#267]
   +- 'Filter ('primary_fuel = Hydro)
      +- 'UnresolvedRelation [plantas], [], false

== Analyzed Logical Plan ==
country_long: string, num_plantas: bigint, capacidad_total_mw: double
Sort [capacidad_total_mw#267 DESC NULLS LAST], true
+- Aggregate [country_long#18], [country_long#18, count(1) AS num_plantas#266L, round(sum(capacity_mw#21), 2) AS capacidad_total_mw#267]
   +- Filter (primary_fuel#24 = Hydro)
      +- SubqueryAlias plantas
         +- View (`plantas`, [country#17,country_long#18,name#19,gppd_idnr#20,capacity_mw#21,latitude#22,longitude#23,primary_fuel#24,other_fuel1#25,other_fuel2#26,other_fuel3#27,commissioning_year#28,owner#29,source#30,url#31,geolocation_source#32,wepp_id#33,year_of_capacity_data#34,generation_gwh_2013#35,generation_gwh_2014#36,generation_gw

**Registrar en la hoja (fila 2):** columnas listadas en `ReadSchema`, de las 36 del archivo · contenido de `PushedFilters` · número de nodos `Exchange` y tipo de cada uno · valor de `isFinalPlan` en la primera línea del plan físico.

*Fuente:* salida de la celda anterior.

**Captura C1:** la salida de `explain(True)`. Basta con la sección `== Physical Plan ==`: no es necesario incluir las tres fases anteriores, que ocupan varias pantallas.

---
## Ejecución 3 — El plan que realmente se ejecutó

Ahora se ejecuta la consulta anterior, sin modificarla.

Se usa `collect()` y no `show()` de forma deliberada: `show(n)` aplica internamente un límite de filas y **cambia el plan**, mientras que `collect()` ejecuta exactamente el plan que se acaba de examinar. Así, lo que aparezca en el portal es comparable, línea por línea, con la captura C1.

El resultado tiene pocas filas (un país por fila), de modo que traerlo al *driver* no representa ningún riesgo.

In [5]:
sc.setJobGroup("E3", "E3 - Ejecucion de la consulta base con collect()")

filas = consulta.collect()

print("Filas devueltas al driver:", len(filas))
print("Primera fila:", filas[0])

Filas devueltas al driver: 128
Primera fila: Row(country_long='China', num_plantas=947, capacidad_total_mw=259025.6)


Esta consulta tiene **dos** `Exchange`, de modo que en el plan ejecutado aparecen **dos nodos `AQEShuffleRead`**, uno por cada uno. Conviene indicar en la hoja a cuál corresponde cada valor registrado.

**Dónde está cada valor en la página de detalle de la consulta**

La página tiene dos zonas, y los valores de esta fila se reparten entre ambas:

- El **DAG**, en la parte superior. Cada recuadro de nodo muestra sus métricas. En los recuadros `AQEShuffleRead` aparecen `number of partitions`, `number of coalesced partitions` y `partition data size`.
- La sección **Details**, debajo del DAG. Muestra el plan en texto, dividido en `== Final Plan ==` y `== Initial Plan ==`, con las `Statistics` de cada `ShuffleQueryStage` y la línea `isFinalPlan` dentro del nodo `AdaptiveSparkPlan`.

Comparar esas dos secciones del texto es el ejercicio central de esta ejecución: el *Initial Plan* es el que se capturó en C1, y el *Final Plan* es el que efectivamente corrió.

**Los dos nodos que solo existen en el Final Plan**

Cada `Exchange` del plan inicial sigue estando en el plan final: el *shuffle* no se elimina. Lo que cambia es que ahora aparecen, por encima de él, dos nodos que el Initial Plan no tiene:

- `ShuffleQueryStage`: representa el *shuffle* ya materializado. Es el nodo que reporta las `Statistics` con el tamaño y el número de filas **reales**, no estimados.
- `AQEShuffleRead`: es la lectura de ese *shuffle*. Su campo `Arguments` indica qué decidió AQE al leerlo; el valor `coalesced` significa que fusionó las particiones.

Esa pareja de nodos es la huella de AQE en el texto del plan, y es lo que permite distinguir de un vistazo el plan estimado del plan realmente ejecutado.

**Cómo ubicar el stage posterior al último *shuffle***

En la pestaña *Stages*, esta consulta aparece en varias filas con la etiqueta `E3`. El stage buscado es el de **mayor Stage Id** entre ellas, y se reconoce por una característica que no comparte con ninguno de los anteriores: **tiene un valor en la columna `Shuffle Read` y ninguno en `Shuffle Write`**. La razón es que su resultado no alimenta otro *shuffle*, sino que se entrega al *driver*.

Conviene verificar, además, que su `Shuffle Read` coincide con el `Shuffle Write` del stage previo: son los dos extremos del mismo intercambio, como se trabajó en el Taller 6.

**Registrar en la hoja (fila 3):** valor de `isFinalPlan` · `sizeInBytes` y `rowCount` de cada `ShuffleQueryStage` · `number of partitions` y `partition data size` de los nodos `AQEShuffleRead` · número de tareas del stage posterior al último *shuffle* · cuántos Job ID tiene la consulta.

*Fuente:* pestaña **SQL / DataFrame** del portal (DAG, sección *Details* y columna *Job IDs*) y pestaña **Stages**.

**Captura C2:** el detalle de la consulta en la pestaña *SQL / DataFrame*, con el plan final y los nodos `AQEShuffleRead` visibles, y con la barra de direcciones a la vista.

*Comparación obligatoria:* el plan de C1 anunciaba un número fijo de particiones después de cada `Exchange`. El plan ejecutado muestra otro. La diferencia entre ambos es AQE trabajando con estadísticas reales, no estimadas.

---
## Ejecución 3b — Una medición repetida en dos condiciones

En la Sesión 23 se midió el número de particiones del resultado de un `GROUP BY` y el valor observado fue **200**, es decir, el valor por defecto sin reducción alguna. Ese resultado es correcto y se puede reproducir. La explicación que se dio en su momento —que el dataset es demasiado pequeño para que AQE intervenga— resulta limitada al contrastarla con una ejecución controlada. En consecuencia, con el objetivo de ampliar las características de su implementación, se repite aquí la misma medición, sobre la misma consulta, cambiando **una sola condición**: la presencia de `cache()`.

In [6]:
sc.setJobGroup("E3b", "E3b - Particiones del resultado SIN cache")

resultado_sin_cache = spark.sql("""
    SELECT country_long, COUNT(*) AS num_plantas
    FROM plantas
    GROUP BY country_long
""")

print("Particiones SIN cache:", resultado_sin_cache.rdd.getNumPartitions())

Particiones SIN cache: 1


In [7]:
sc.setJobGroup("E3c", "E3c - Particiones del mismo resultado CON cache")

resultado_con_cache = spark.sql("""
    SELECT country_long, COUNT(*) AS num_plantas
    FROM plantas
    GROUP BY country_long
""")

resultado_con_cache.cache()
resultado_con_cache.count()

print("Particiones CON cache:", resultado_con_cache.rdd.getNumPartitions())

resultado_con_cache.unpersist()
print("Cache liberado.")

[Stage 24:================================================>     (181 + 2) / 200]

Particiones CON cache: 200
Cache liberado.


**Qué explica la diferencia**

Con datos pequeños AQE **sí** interviene, y de forma agresiva: al terminar el *shuffle* comprueba que las particiones resultantes son diminutas y las fusiona (*coalescing*). Por eso la versión sin `cache()` termina con muy pocas particiones.

Lo que bloquea esa fusión en el otro caso (Sesión 23) es el `cache()` mismo. Desde Spark 3.5, la configuración `spark.sql.optimizer.canChangeCachedPlanOutputPartitioning` está habilitada por defecto, de modo que AQE sí se aplica a un plan que va a ser almacenado en caché; sin embargo, en esta versión AQE **no fusiona la etapa final** de ese plan. Como la consulta tiene un único *shuffle*, la etapa posterior a ese *shuffle* es precisamente la etapa final, y por eso conserva las 200 particiones.

La distinción quedó documentada de forma explícita en la incidencia **SPARK-46995**, resuelta para Spark 4.0.0, que redefine esa configuración: en la nueva semántica, el valor `true` autoriza la fusión de la etapa final, y el valor `false` mantiene el comportamiento anterior —AQE activo, sin fusión de la etapa final—, que es el de Spark 3.5.7, la versión instalada en este clúster.

**Lo que no cambia:** la afirmación de la Sesión 23 sobre la importancia de AQE en producción sigue siendo válida. Lo que se corrige es la causa atribuida a ese 200 concreto.

**Registrar en la hoja (fila 4):** particiones sin `cache()` · particiones con `cache()`.

*Fuente:* salida de las dos celdas anteriores.

*No requiere captura.*

---
## Ejecución 4 — Ajuste 1: reducir el trabajo de ordenamiento

La consulta de la Ejecución 2 ordena **todos** los países por capacidad. Ese ordenamiento global obliga a un `Exchange rangepartitioning`: para ordenar de extremo a extremo, Spark debe redistribuir las filas entre executors por rangos de valor.

Sin embargo, la pregunta real casi nunca es "el orden completo", sino "los primeros". Al añadir `LIMIT`, Catalyst puede resolverlo de otra manera: cada partición ordena localmente, conserva solo sus mejores filas y el *driver* combina esos resultados parciales. El operador que aparece en el plan cuando esto ocurre es `TakeOrderedAndProject`, y reemplaza a la pareja `Sort` + `Exchange rangepartitioning`.

Es un ajuste en la **consulta**, no en la configuración del clúster: el mismo dato, preguntado de otra forma, cuesta un *shuffle* menos.

In [8]:
sc.setJobGroup("E4", "E4 - Ajuste 1: ORDER BY con LIMIT")

consulta_top = spark.sql("""
    SELECT country_long,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    WHERE primary_fuel = 'Hydro'
    GROUP BY country_long
    ORDER BY capacidad_total_mw DESC
    LIMIT 10
""")

consulta_top.explain()

filas_top = consulta_top.collect()
print("Filas devueltas al driver:", len(filas_top))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=10, orderBy=[capacidad_total_mw#406 DESC NULLS LAST], output=[country_long#18,num_plantas#405L,capacidad_total_mw#406])
   +- HashAggregate(keys=[country_long#18], functions=[count(1), sum(capacity_mw#21)])
      +- Exchange hashpartitioning(country_long#18, 200), ENSURE_REQUIREMENTS, [plan_id=305]
         +- HashAggregate(keys=[country_long#18], functions=[partial_count(1), partial_sum(capacity_mw#21)])
            +- Project [country_long#18, capacity_mw#21]
               +- Filter (isnotnull(primary_fuel#24) AND (primary_fuel#24 = Hydro))
                  +- FileScan csv [country_long#18,capacity_mw#21,primary_fuel#24] Batched: false, DataFilters: [isnotnull(primary_fuel#24), (primary_fuel#24 = Hydro)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://hadoop-virtualbox:9000/datasets/global_power_plant_database.csv], PartitionFilters: [], PushedFilters: [IsNotNull(primary_fuel), EqualTo(

**Registrar en la hoja (fila 5):** operador que aparece en lugar de `Sort` + `Exchange rangepartitioning` · número de nodos `Exchange` que quedan en el plan, frente a los de la fila 2 · cuántos Job ID tiene la consulta y cuántos stages tiene cada uno.

*Fuente:* salida de la celda anterior, pestaña **SQL / DataFrame** y pestaña **Stages**.

**Captura C3:** el detalle de la consulta `E4` en la pestaña *SQL / DataFrame*, con la barra de direcciones a la vista.

---
## Ejecución 5 — Ajuste 2: la misma consulta sobre otro formato

El último ajuste no toca la consulta: cambia **de dónde se leen los datos**. Se ejecuta la consulta de la Ejecución 4 dos veces, primero sobre la vista `plantas` (CSV) y luego sobre `plantas_parquet` (Parquet), que contiene exactamente los mismos registros.

La primera repite de forma deliberada la consulta de E4: así las dos lecturas quedan una junto a la otra y pueden compararse en una misma captura.

En el plan conviene observar el campo `Batched` dentro de `FileScan`, además de `Format` y `ReadSchema`. Como se vio en la Sesión 23, ambos formatos muestran `PushedFilters` no vacíos, pero solo uno de los dos puede aprovecharlos para evitar leer bloques completos del disco.

In [9]:
sc.setJobGroup("E5a", "E5a - Consulta sobre CSV")

consulta_csv = spark.sql("""
    SELECT country_long,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    WHERE primary_fuel = 'Hydro'
    GROUP BY country_long
    ORDER BY capacidad_total_mw DESC
    LIMIT 10
""")

print("=== Plan sobre CSV ===")
consulta_csv.explain()
consulta_csv.collect()
print("Ejecucion sobre CSV terminada.")

=== Plan sobre CSV ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=10, orderBy=[capacidad_total_mw#422 DESC NULLS LAST], output=[country_long#18,num_plantas#421L,capacidad_total_mw#422])
   +- HashAggregate(keys=[country_long#18], functions=[count(1), sum(capacity_mw#21)])
      +- Exchange hashpartitioning(country_long#18, 200), ENSURE_REQUIREMENTS, [plan_id=364]
         +- HashAggregate(keys=[country_long#18], functions=[partial_count(1), partial_sum(capacity_mw#21)])
            +- Project [country_long#18, capacity_mw#21]
               +- Filter (isnotnull(primary_fuel#24) AND (primary_fuel#24 = Hydro))
                  +- FileScan csv [country_long#18,capacity_mw#21,primary_fuel#24] Batched: false, DataFilters: [isnotnull(primary_fuel#24), (primary_fuel#24 = Hydro)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://hadoop-virtualbox:9000/datasets/global_power_plant_database.csv], PartitionFilters: [], PushedFilters: [IsNotNull(

In [10]:
sc.setJobGroup("E5b", "E5b - Misma consulta sobre Parquet")

consulta_parquet = spark.sql("""
    SELECT country_long,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas_parquet
    WHERE primary_fuel = 'Hydro'
    GROUP BY country_long
    ORDER BY capacidad_total_mw DESC
    LIMIT 10
""")

print("=== Plan sobre Parquet ===")
consulta_parquet.explain()
consulta_parquet.collect()
print("Ejecucion sobre Parquet terminada.")

=== Plan sobre Parquet ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=10, orderBy=[capacidad_total_mw#438 DESC NULLS LAST], output=[country_long#163,num_plantas#437L,capacidad_total_mw#438])
   +- HashAggregate(keys=[country_long#163], functions=[count(1), sum(capacity_mw#166)])
      +- Exchange hashpartitioning(country_long#163, 200), ENSURE_REQUIREMENTS, [plan_id=423]
         +- HashAggregate(keys=[country_long#163], functions=[partial_count(1), partial_sum(capacity_mw#166)])
            +- Project [country_long#163, capacity_mw#166]
               +- Filter (isnotnull(primary_fuel#169) AND (primary_fuel#169 = Hydro))
                  +- FileScan parquet [country_long#163,capacity_mw#166,primary_fuel#169] Batched: true, DataFilters: [isnotnull(primary_fuel#169), (primary_fuel#169 = Hydro)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://hadoop-virtualbox:9000/datasets/plantas_parquet], PartitionFilters: [], PushedFilters: [

[Stage 34:=============================>                            (1 + 1) / 2]

Ejecucion sobre Parquet terminada.


**Registrar en la hoja (fila 6):** `Format` y `Batched` en el `FileScan` de cada consulta · columnas listadas en `ReadSchema` en cada caso.

*Fuente:* salida de las dos celdas anteriores.

**Captura C4:** las dos salidas de `explain()`, con las dos líneas `FileScan` visibles. Si no caben en una sola imagen, se admiten dos imágenes numeradas C4a y C4b.

---
## Cierre de la Fase 1

Antes del corte, verificar que:

1. La hoja de registro está completa, con la fuente de cada valor y el número de captura donde corresponde.
2. El archivo de Word contiene las cuatro capturas, numeradas C1 a C4.
3. El archivo se entregó por Microsoft Teams.

**Durante la Fase 2:**

- **No ejecutar `spark.stop()`** ni apagar el kernel (*Shut Down Kernel*). La aplicación debe seguir viva para poder consultar el portal.
- **Cerrar la pestaña de este notebook.** Cerrar la pestaña no detiene la aplicación: el kernel sigue activo en el servidor y el portal de Spark continúa respondiendo con normalidad.
- El material permitido durante la Fase 2 está listado en el documento del taller.

In [ ]:
# ATENCION: esta linea debe permanecer comentada durante todo el taller.
# Ejecutarla cierra la aplicacion y deja el portal de Spark inaccesible,
# lo que impide responder el cuestionario de la Fase 2.
# spark.stop()